In [46]:
'''
            UŻYCIE AI

Kod został napisany z wykorzystaniem 
generatywnej AI. Autorzy zweryfikowali
wszystko i prezentują własny wkład intelektualny,
biorąc pełną odpowiedzialność za ostateczną postać pracy
'''

'\n            UŻYCIE AI\n\nKod został napisany z wykorzystaniem \ngeneratywnej AI. Autorzy zweryfikowali\nwszystko i prezentują własny wkład intelektualny,\nbiorąc pełną odpowiedzialność za ostateczną postać pracy\n'

In [47]:
# dropna przy value_counts można usunąć, bo jest sprawdzone wcześniej
# zrobienie odpowiednich wartosci w polach mozna zrobic od razu przy 
#     wczytywaniu, zamiast potem sprawdzac i usuwac

In [48]:
import numpy as np
import pandas as pd

In [49]:
# wczytanie lookupu stref
zones = pd.read_csv("taxi_zone_lookup.csv")

# zostawiamy tylko strefy z Manhattanu
manhattan_zones = zones.loc[zones["Borough"] == "Manhattan", "LocationID"].tolist()

manhattan_zones;

In [50]:
''' 
    WCZYTANIE TAXI. FITLROWANIE. 
    
    Wyrzucamy kolumny "VendorID" i "store_and_fwd_flag".
    Zostawiamy tylko te przejazdy, które:
    - zaczynają i kończą się na Manhattanie. 
    - mają dodatnie / nieujemne wartości w 
        polach gdzie inne nie mają sensu
'''

taxi = (pd.read_parquet("yellow_tripdata_2025-01.parquet")
      .drop(columns = ["VendorID", "store_and_fwd_flag"])
      .query('PULocationID in @manhattan_zones and '
             'DOLocationID in @manhattan_zones and '
             'passenger_count >= 0 and '
             'trip_distance > 0 and '
             'fare_amount > 0 and '
             'extra >= 0 and '
             'mta_tax >= 0 and '
             'tip_amount >= 0 and '
             'tolls_amount >= 0 and '
             'improvement_surcharge > 0 and '
             'total_amount >= 0 and '
             'congestion_surcharge >= 0 and '
             'Airport_fee >= 0 and '
             'cbd_congestion_fee >= 0') 
      )

In [51]:
taxi

,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,229,237,1,10.0,3.50,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.00
1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,236,237,1,5.1,3.50,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.00
2,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,141,141,1,5.1,3.50,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.00
3,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,244,244,2,7.2,1.00,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.00
4,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,244,116,2,5.8,1.00,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2935072,2025-01-31 23:01:14,2025-01-31 23:08:49,1.0,1.50,1.0,163,48,2,8.6,4.25,0.5,0.00,0.0,1.0,14.35,2.5,0.0,0.75
2935073,2025-01-31 23:11:28,2025-01-31 23:14:45,1.0,0.60,1.0,48,48,2,5.1,4.25,0.5,0.00,0.0,1.0,10.85,2.5,0.0,0.75
2935074,2025-01-31 23:14:32,2025-01-31 23:28:02,1.0,1.96,1.0,142,164,1,14.2,1.00,0.5,2.00,0.0,1.0,21.95,2.5,0.0,0.75
2935075,2025-01-31 23:15:51,2025-01-31 23:21:22,2.0,0.89,1.0,230,142,1,7.2,1.00,0.5,2.59,0.0,1.0,15.54,2.5,0.0,0.75


In [52]:
# Sprawdzamy czy są jakieś brakujące wartości
taxi.isna().sum()
# nie ma

tpep_pickup_datetime     0
tpep_dropoff_datetime    0
passenger_count          0
trip_distance            0
RatecodeID               0
PULocationID             0
DOLocationID             0
payment_type             0
fare_amount              0
extra                    0
mta_tax                  0
tip_amount               0
tolls_amount             0
improvement_surcharge    0
total_amount             0
congestion_surcharge     0
Airport_fee              0
cbd_congestion_fee       0
dtype: int64

In [53]:
# Sprawdzamy wartości congestion_surcharge
taxi["congestion_surcharge"].value_counts(dropna=False)
# zgadzają się z 
#     https://www.nyc.gov/site/tlc/about/congestion-surcharge.page

congestion_surcharge
2.5    2375439
0.0      35228
Name: count, dtype: int64

In [54]:
# Sprawdzamy wartości cbd_congestion_fee
taxi["cbd_congestion_fee"].value_counts(dropna=False)
# powinny zgadzać się z https://congestionreliefzone.mta.info/tolling

cbd_congestion_fee
0.75    1645680
0.00     764987
Name: count, dtype: int64

In [55]:
# Usuwamy inne niż 0 i 0.75
taxi = taxi[taxi["cbd_congestion_fee"].isin([0, 0.75])]
taxi["cbd_congestion_fee"].value_counts(dropna=False)

cbd_congestion_fee
0.75    1645680
0.00     764987
Name: count, dtype: int64

In [56]:
# Sprawdzamy wartości airport_fee
taxi["Airport_fee"].value_counts(dropna=False)
# jest trochę niezerowych

Airport_fee
0.00    2410573
1.75         94
Name: count, dtype: int64

In [57]:
# Usuwamy niezerowe
taxi = taxi[taxi["Airport_fee"] == 0]
taxi["Airport_fee"].value_counts()

Airport_fee
0.0    2410573
Name: count, dtype: int64

In [58]:
# Sprawdzamy wartości RatecodeID
taxi["RatecodeID"].value_counts()

RatecodeID
1.0     2405923
5.0        2456
2.0        1318
99.0        459
3.0         392
4.0          24
6.0           1
Name: count, dtype: int64

In [59]:
# zostawiamy tylko standard, bo reszta nas nie dotyczy
#     https://www.nyc.gov/site/tlc/passengers/taxi-fare.page
taxi = taxi[~taxi["RatecodeID"].isin([99, 6, 5, 4, 3, 2])]
taxi["RatecodeID"].value_counts()

RatecodeID
1.0    2405923
Name: count, dtype: int64

In [60]:
# Sprawdzamy wartości payment_type
taxi["payment_type"].value_counts()

payment_type
1    2068089
2     302824
4      25832
3       9178
Name: count, dtype: int64

In [61]:
# usuwamy no charge i dispute
taxi = taxi[~taxi["payment_type"].isin([3, 4])]
taxi["payment_type"].value_counts()

payment_type
1    2068089
2     302824
Name: count, dtype: int64

In [62]:
# Sprawdzamy wartości improvement_surcharge
taxi["improvement_surcharge"].value_counts()

improvement_surcharge
1.0    2370913
Name: count, dtype: int64

In [63]:
# usuwamy zerowe
taxi = taxi[taxi["improvement_surcharge"] != 0]
taxi["improvement_surcharge"].value_counts()

improvement_surcharge
1.0    2370913
Name: count, dtype: int64

In [64]:
# Sprawdzamy wartości mta_tax
taxi["mta_tax"].value_counts()

mta_tax
0.5    2370482
0.0        431
Name: count, dtype: int64

In [65]:
# usuwamy zerowe
taxi = taxi[taxi["mta_tax"] != 0]
taxi["mta_tax"].value_counts()

mta_tax
0.5    2370482
Name: count, dtype: int64

In [66]:
# Sprawdzamy wartości extra
taxi["extra"].value_counts();
# to jest raczej ok, chociaż dziwne możnaby usunąć

In [67]:
# Sprawdzamy wartości fare_amount
print(taxi["fare_amount"].value_counts().sort_index())
#print(taxi["fare_amount"].value_counts().sort_index().head(10))
#print(taxi["fare_amount"].value_counts().sort_index().tail(10))
# wszystkie wydają się mieć sens teoretyczny, ale 

fare_amount
3.0       1982
3.7       8258
4.0          1
4.4      31386
4.8          6
         ...  
380.3        1
434.9        1
441.9        1
484.6        1
498.6        1
Name: count, Length: 747, dtype: int64


In [68]:
# Sprawdzamy rozkład do przedziałów
mniej_3 = (taxi["fare_amount"] < 3).sum()
miedzy_3_85 = ((taxi["fare_amount"] >= 3) & 
              (taxi["fare_amount"] <= 85)).sum()
miedzy_85_150 = ((taxi["fare_amount"] > 85) & 
                (taxi["fare_amount"] <= 150)).sum()
wieksze_150 = (taxi["fare_amount"] > 150).sum()

print('"< 3": ', mniej_3)
print('"<3 ; 85>" :', miedzy_3_85)
print('(85 ; 150>" :', miedzy_85_150)
print('"> 150": ', wieksze_150)
# do sprawdzenia czy zaburzaja oszacowanie i ew. usunięcia

"< 3":  0
"<3 ; 85>" : 2370365
(85 ; 150>" : 85
"> 150":  32


In [69]:
# Sprawdzamy wartości trip_distance
taxi["trip_distance"].value_counts().sort_index()
# wszystkie wydają się mieć sens teoretyczny, ale 

trip_distance
0.01       429
0.02       330
0.03       282
0.04       224
0.05       239
          ... 
90.84        1
102.20       1
1472.37      1
1847.61      1
2001.95      1
Name: count, Length: 1510, dtype: int64

In [70]:
# Sprawdzamy rozkład do przedziałów
mniej_02 = (taxi["trip_distance"] < 0.2).sum()
miedzy_02_20 = ((taxi["trip_distance"] >= 0.2) & 
              (taxi["trip_distance"] <= 20)).sum()
miedzy_20_35 = ((taxi["trip_distance"] > 20) & 
                (taxi["trip_distance"] <= 35)).sum()
wieksze_35 = (taxi["trip_distance"] > 35).sum()

print('"< 0.2": ', mniej_02)
print('"<0.2 ; 20>" :', miedzy_02_20)
print('(20 ; 35>" :', miedzy_20_35)
print('"> 35": ', wieksze_35)
# do sprawdzenia czy zaburzaja oszacowanie i ew. usunięcia

"< 0.2":  6379
"<0.2 ; 20>" : 2364026
(20 ; 35>" : 60
"> 35":  17


In [71]:
# Dodajemy zmienna trip_time (czas w sekundach)
#     i trip_time_mins (czas w minutach)
taxi["trip_time"] = (
    taxi["tpep_dropoff_datetime"] - taxi["tpep_pickup_datetime"]
    ).dt.total_seconds()
taxi["trip_time_mins"] = taxi["trip_time"] / 60

In [72]:
taxi.head(10)

,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,trip_time,trip_time_mins
0,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0,501.0,8.350000
1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0,153.0,2.550000
2,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0,117.0,1.950000
3,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0,334.0,5.566667
4,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0,212.0,3.533333
5,2025-01-01 00:48:24,2025-01-01 01:08:26,2.0,2.63,1.0,239,68,2,19.1,1.0,0.5,0.00,0.0,1.0,24.10,2.5,0.0,0.0,1202.0,20.033333
6,2025-01-01 00:14:47,2025-01-01 00:16:15,0.0,0.40,1.0,170,170,1,4.4,3.5,0.5,2.35,0.0,1.0,11.75,2.5,0.0,0.0,88.0,1.466667
7,2025-01-01 00:39:27,2025-01-01 00:51:51,0.0,1.60,1.0,234,148,1,12.1,3.5,0.5,2.00,0.0,1.0,19.10,2.5,0.0,0.0,744.0,12.400000
8,2025-01-01 00:53:43,2025-01-01 01:13:23,0.0,2.80,1.0,148,170,1,19.1,3.5,0.5,3.00,0.0,1.0,27.10,2.5,0.0,0.0,1180.0,19.666667
9,2025-01-01 00:00:02,2025-01-01 00:09:36,1.0,1.71,1.0,237,262,2,11.4,1.0,0.5,0.00,0.0,1.0,16.40,2.5,0.0,0.0,574.0,9.566667


In [73]:
# Sprawdzamy wartości trip_time
taxi["trip_time"].value_counts().sort_index()

trip_time
-3088339.0       1
 0.0          1134
 1.0             1
 2.0             2
 3.0             7
              ... 
 86363.0         1
 86366.0         1
 86367.0         1
 86386.0         1
 93919.0         1
Name: count, Length: 4625, dtype: int64

In [74]:
# usuwamy bezsensowne
taxi = taxi[taxi["trip_time"] > 0]
taxi["trip_time"].value_counts().sort_index()
# teraz wszystkie wydają się mieć sens teoretyczny, ale

trip_time
1.0         1
2.0         2
3.0         7
4.0        22
5.0        25
           ..
86363.0     1
86366.0     1
86367.0     1
86386.0     1
93919.0     1
Name: count, Length: 4623, dtype: int64

In [75]:
# Sprawdzamy rozkład do przedziałów
mniej_60 = (taxi["trip_time"] < 60).sum()
miedzy_60_1800 = ((taxi["trip_time"] >= 60) & 
                 (taxi["trip_time"] <= 1800)).sum()
miedzy_1800_2700 = ((taxi["trip_time"] > 1800) & 
                   (taxi["trip_time"] <= 2700)).sum()
wieksze_2700 = (taxi["trip_time"] > 2700).sum()

print('"< 60": ', mniej_60)
print('"<60 ; 1800>" :', miedzy_60_1800)
print('(1800 ; 2700>" :', miedzy_1800_2700)
print('"> 2700": ', wieksze_2700)
# do sprawdzenia czy zaburzaja oszacowanie i ew. usunięcia

"< 60":  2666
"<60 ; 1800>" : 2329538
(1800 ; 2700>" : 34056
"> 2700":  3087


In [76]:
# Sprawdzamy liczbę ,,sensownych'' i nie
mask = (taxi["fare_amount"].between(3, 85) &
        taxi["trip_distance"].between(0.2, 35) &
        taxi["trip_time"].between(60, 2700))

print("sensowne: ", mask.sum()) 
print("bezsensowne: ", (~mask).sum())
# prawie wszystkie mają sens, reszta do sprawdzenia

sensowne:  2359031
bezsensowne:  10316


In [77]:
'''# Wstępnie przefiltrowane dane zapisujemy do porównań
taxi_wstępne = taxi.copy()'''

'# Wstępnie przefiltrowane dane zapisujemy do porównań\ntaxi_wstępne = taxi.copy()'

In [78]:
'''
    TERAZ, PO WSTĘPNYM PRZEFILITROWANIU DANYCH I
    OKREŚLENIU, ŻE WSZYSTKIE MAJĄ SENS TEORETYCZNY,
    A ZDECYDOWANA WIĘKSZOŚC TAKŻE SENS REALNY,
    MOŻEMY PRZEJŚĆ DO DALSZEJ ANALIZY ORAZ JUŻ 
    POD MODEL EKONOMETRYCZNY
'''

'\n    TERAZ, PO WSTĘPNYM PRZEFILITROWANIU DANYCH I\n    OKREŚLENIU, ŻE WSZYSTKIE MAJĄ SENS TEORETYCZNY,\n    A ZDECYDOWANA WIĘKSZOŚC TAKŻE SENS REALNY,\n    MOŻEMY PRZEJŚĆ DO DALSZEJ ANALIZY ORAZ JUŻ \n    POD MODEL EKONOMETRYCZNY\n'

In [79]:
taxi;

In [80]:
# Wyciągamy z tpep_pickup_datetime godzinę i dzień tygodnia
#     przypisujemy godzinę na zasadnie + / - pół godziny
taxi = taxi.assign(hour = taxi["tpep_pickup_datetime"].dt.hour,
                   weekday = taxi["tpep_pickup_datetime"].dt.weekday,
                   month = taxi["tpep_pickup_datetime"].dt.month)
# weekday 0 to poniedziałek

In [81]:
taxi

,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,...,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,trip_time,trip_time_mins,hour,weekday,month
0,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,229,237,1,10.0,3.50,...,1.0,18.00,2.5,0.0,0.00,501.0,8.350000,0,2,1
1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,236,237,1,5.1,3.50,...,1.0,12.12,2.5,0.0,0.00,153.0,2.550000,0,2,1
2,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,141,141,1,5.1,3.50,...,1.0,12.10,2.5,0.0,0.00,117.0,1.950000,0,2,1
3,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,244,244,2,7.2,1.00,...,1.0,9.70,0.0,0.0,0.00,334.0,5.566667,0,2,1
4,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,244,116,2,5.8,1.00,...,1.0,8.30,0.0,0.0,0.00,212.0,3.533333,0,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2935072,2025-01-31 23:01:14,2025-01-31 23:08:49,1.0,1.50,1.0,163,48,2,8.6,4.25,...,1.0,14.35,2.5,0.0,0.75,455.0,7.583333,23,4,1
2935073,2025-01-31 23:11:28,2025-01-31 23:14:45,1.0,0.60,1.0,48,48,2,5.1,4.25,...,1.0,10.85,2.5,0.0,0.75,197.0,3.283333,23,4,1
2935074,2025-01-31 23:14:32,2025-01-31 23:28:02,1.0,1.96,1.0,142,164,1,14.2,1.00,...,1.0,21.95,2.5,0.0,0.75,810.0,13.500000,23,4,1
2935075,2025-01-31 23:15:51,2025-01-31 23:21:22,2.0,0.89,1.0,230,142,1,7.2,1.00,...,1.0,15.54,2.5,0.0,0.75,331.0,5.516667,23,4,1


In [82]:
taxi['month'].value_counts()

month
1     2369331
12         15
2           1
Name: count, dtype: int64

In [83]:
# Usuwamy rekordy z błędnymi miesiącami
taxi = taxi[taxi["month"] == 1]

In [84]:
taxi['month'].value_counts()

month
1    2369331
Name: count, dtype: int64

In [85]:
# Usuwamy kolumny niepotrzebne do modelu
taxi = taxi.drop(columns = ["tpep_dropoff_datetime", "passenger_count",
                          "RatecodeID", "payment_type", "extra",
                          "mta_tax", "tip_amount", "tolls_amount",
                          "improvement_surcharge", "congestion_surcharge",
                          "Airport_fee", "cbd_congestion_fee",
                          "PULocationID", "DOLocationID"])

# Dodajemy kolumnę type, żeby można było połączyć z fhv
taxi["type"] = "Taxi"

In [86]:
taxi

,tpep_pickup_datetime,trip_distance,fare_amount,total_amount,trip_time,trip_time_mins,hour,weekday,month,type
0,2025-01-01 00:18:38,1.60,10.0,18.00,501.0,8.350000,0,2,1,Taxi
1,2025-01-01 00:32:40,0.50,5.1,12.12,153.0,2.550000,0,2,1,Taxi
2,2025-01-01 00:44:04,0.60,5.1,12.10,117.0,1.950000,0,2,1,Taxi
3,2025-01-01 00:14:27,0.52,7.2,9.70,334.0,5.566667,0,2,1,Taxi
4,2025-01-01 00:21:34,0.66,5.8,8.30,212.0,3.533333,0,2,1,Taxi
...,...,...,...,...,...,...,...,...,...,...
2935072,2025-01-31 23:01:14,1.50,8.6,14.35,455.0,7.583333,23,4,1,Taxi
2935073,2025-01-31 23:11:28,0.60,5.1,10.85,197.0,3.283333,23,4,1,Taxi
2935074,2025-01-31 23:14:32,1.96,14.2,21.95,810.0,13.500000,23,4,1,Taxi
2935075,2025-01-31 23:15:51,0.89,7.2,15.54,331.0,5.516667,23,4,1,Taxi


In [87]:
'''# Eksportujemy taxi do csv
taxi.to_csv("taxi_do_statOp.csv", index = False)'''

'# Eksportujemy taxi do csv\ntaxi.to_csv("taxi_do_statOp.csv", index = False)'

In [88]:
# Łączymy też taxi z fhv i eksportujemy do csv
#     może być wygodniej do statystyki opisowej
fhv = pd.read_csv("fhv_do_statOp.csv")

# Robimy wspólne kolumny
fhv_common = fhv.rename(columns={
    "pickup_datetime": "datetime",
    "trip_miles": "distance",
    "base_passenger_fare": "base_fare",
    "hvfhs_license_num": "provider",
})
fhv_common["provider"] = fhv_common["provider"].map({
    "HV0003": "Uber",
    "HV0005": "Lyft"
})

taxi_common = taxi.rename(columns={
    "tpep_pickup_datetime": "datetime",
    "trip_distance": "distance",
    "fare_amount": "base_fare",
    "type": "provider",
})

# Zostawiamy tylko wspolne kolumny
common_cols = [
    "provider", "datetime", "distance",
    "trip_time", "trip_time_mins",
    "base_fare", "total_amount",
    "hour", "weekday", "month"
]

fhv_common = fhv_common[common_cols]
taxi_common = taxi_common[common_cols]

# Łączymy w jeden zbior
wszystkie = pd.concat([fhv_common, taxi_common], ignore_index=True)

In [89]:
wszystkie

,provider,datetime,distance,trip_time,trip_time_mins,base_fare,total_amount,hour,weekday,month
0,Uber,2025-01-01 00:33:25,1.320,1259.0,20.983333,18.21,22.92,0,2,1
1,Lyft,2025-01-01 00:29:49,3.313,723.0,12.050000,16.97,18.95,0,2,1
2,Lyft,2025-01-01 00:12:35,7.254,2342.0,39.033333,33.49,39.99,0,2,1
3,Lyft,2025-01-01 00:06:32,2.095,632.0,10.533333,12.57,16.78,0,2,1
4,Lyft,2025-01-01 00:19:16,2.876,573.0,9.550000,11.86,15.99,0,2,1
...,...,...,...,...,...,...,...,...,...,...
7731183,Taxi,2025-01-31 23:01:14,1.500,455.0,7.583333,8.60,14.35,23,4,1
7731184,Taxi,2025-01-31 23:11:28,0.600,197.0,3.283333,5.10,10.85,23,4,1
7731185,Taxi,2025-01-31 23:14:32,1.960,810.0,13.500000,14.20,21.95,23,4,1
7731186,Taxi,2025-01-31 23:15:51,0.890,331.0,5.516667,7.20,15.54,23,4,1


In [90]:
# Eksportujemy do csv
wszystkie.to_csv("wszystkie_do_statOp.csv", index=False)